# Kokoro TTS Audiobook Generator
This notebook is for converting any structured JSON file (from DOCX, PDF, or other sources) into a high-quality audiobook using the Kokoro TTS pipeline. Each chapter or section is synthesized as a separate FLAC file, and all audio is packaged into a single archive for easy download.

**Workflow:**
1. Download or load the JSON file containing the book/lecture structure.
2. Install and import required libraries.
3. Clean and normalize the text for TTS.
4. Synthesize audio for each chapter/section and save as FLAC files.
5. Archive all audio files and clean up temporary data.

## 1. Download and Load the JSON File
Download or load the structured JSON file (e.g., `Lectures.json`) containing the chapters/sections and their content. The file is loaded directly from a GitHub repository. This step prepares the data for audiobook generation.

In [ ]:
!rm -rf Audiobook
!rm Audiobook.zip
print("Audiobook folder and zip file deleted successfully.")

import requests

url = "https://api.github.com/repos/Kisara-k/kokoro-tts-source/contents/Lectures.json"
headers = {"Accept": "application/vnd.github.v3.raw"}

response = requests.get(url, headers=headers)
response.raise_for_status()

book = response.json()
print("Successfully loaded the audiobook file.")

## 2. Install and Import Required Libraries
Install all necessary Python packages (Kokoro TTS, soundfile, roman, etc.), set up the TTS pipeline, and import supporting libraries for audio processing and display. This step ensures the environment is ready for synthesis and audio file handling.

In [ ]:
%%time
%%capture --no-stderr

import os

# Check if the Kokoro-82M directory exists, if not, clone it
if not os.path.isdir("Kokoro-82M"):
    !git clone https://huggingface.co/hexgrad/Kokoro-82M!pip install -q kokoro soundfile

# Install espeak, used for out-of-dictionary fallback
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!pip install roman

from kokoro import KPipeline
from IPython.display import display, Audio
import soundfile as sf
import numpy as np

pipeline = KPipeline(lang_code='a')

## 3. Clean and Normalize Text
This step defines and applies a cleaning function to normalize and preprocess the text for TTS synthesis. It handles Roman numerals, accents, capitalization, and removes unwanted characters to ensure the generated audio is clear and natural.

In [ ]:
import time
import re
import unicodedata
import roman
import string

def clean(text):
    
    def roman_to_arabic(match):
        numeral = match.group(0)
        # if numeral.upper() in ['I', 'X']:
        #     return numeral
        # if len(numeral) <= 1:
        #     return numeral
        try:
            return str(roman.fromRoman(numeral.upper()))
        except roman.InvalidRomanNumeralError:
            return numeral
    # text = re.sub(r'\b[MCDXLIV]+\b', roman_to_arabic, text)

    def replace_chapter_roman(match):
        chapter_part = match.group(1)              # CHAPTER
        numeral = match.group(2)                   # XV
        converted = roman_to_arabic(re.match(r'.*', numeral))  # wrap numeral in match object
        return chapter_part + converted
    
    text = re.sub(r'(?i)(\bsection\b\s+)([MCDXLIV]+)\b', replace_chapter_roman, text)
    
    def strip_accents(text):
        return ''.join(
            c for c in unicodedata.normalize('NFKD', text)
            if not unicodedata.combining(c)
        )
    text = strip_accents(text)
    
    def remove_capital_words(match):
        word = match.group(0)
        if len(word) <= 3:
            return word  # Keep upto n letter capital words
        return word.title()
    
    text = re.sub(r"\b[A-Z][A-Z'’\-]*\b", remove_capital_words, text)

    def remove_special_unicode(text):
        def is_allowed(c):
            # First check if it's basic ASCII printable
            if c in (string.ascii_letters + string.digits + string.punctuation + " \t\n\r"):
                return True
            # Otherwise, allow if it's punctuation/symbols but not letters from foreign scripts
            category = unicodedata.category(c)
            if category.startswith(('P', 'S')):  # P = punctuation, S = symbol
                return True
            return False
        return ''.join(c for c in text if is_allowed(c))
    
    text = remove_special_unicode(text)

    return text

## 4. Synthesize Audio
For each chapter or section, the application uses the Kokoro TTS pipeline to generate audio, saving each as a FLAC file in the `Audiobook` folder.

List of available voices: https://huggingface.co/hexgrad/Kokoro-82M/blob/main/VOICES.md

In [ ]:
# Select the voice you want to use
# These are my preferred voices, you can change them as needed

voice = "af_bella"
# voice = "am_echo"

# Ensure output folder exists
os.makedirs("Audiobook", exist_ok=True)

# Pre-clean all text upfront
for chapter in book:
    chapter["clean_text"] = clean(chapter["content"]).strip()

def time_delta(start_time):
    elapsed = int(time.time() - start_time)
    minutes, seconds = divmod(elapsed, 60)
    return f"{minutes:02d}:{seconds:02d}"

def est_time(remaining_words, total_words, initial_time):
    words_done = total_words - remaining_words
    elapsed = time.time() - initial_time
    if words_done > 0:
        estimated_total_time = elapsed * total_words / words_done
    else:
        estimated_total_time = 0
    return divmod(int(estimated_total_time), 60)

def save_audio_async(executor, filename, full_audio):
    executor.submit(sf.write, filename, full_audio, 24000, format='FLAC')

# Pre-calculate total word count
total_words = sum(len(chap["clean_text"].split()) for chap in book)
remaining_words = total_words
initial_time = time.time()

# ThreadPoolExecutor for async file saving
from concurrent.futures import ThreadPoolExecutor
executor = ThreadPoolExecutor(max_workers=4)

# Loop through each section/chapter in the book
for chapter in book:
    index = chapter["index"]
    title = chapter["title"].strip().replace("/", "-")
    text = chapter["clean_text"]

    if not text:
        continue  # skip empty content

    chapter_word_count = len(text.split())
    start_time = time.time()

    generator = pipeline(
        text=text,
        voice=voice,
        speed=1,
        split_pattern=None
    )

    # Collect and concatenate audio
    all_audio = [audio for _, _, audio in generator]

    silence = np.zeros(int(24000 * 0.5), dtype=np.float32)
    audios = [silence]
    audios.extend(all_audio)
    audios.extend([silence] * 4)
    full_audio = np.concatenate(audios)

    pad = (24000 - len(full_audio) % 24000) % 24000
    if pad > 0:
        full_audio = np.concatenate([full_audio, np.zeros(pad, dtype=np.float32)])

    name = f"{index:02d} {title}"
    filename = os.path.join("Audiobook", f"{name}.flac")

    save_audio_async(executor, filename, full_audio)

    remaining_words -= chapter_word_count
    est_minutes, est_seconds = est_time(remaining_words, total_words, initial_time)

    print(f"Saved: {name:<50} | in {time_delta(start_time)} | total {time_delta(initial_time)} | ETA {est_minutes:02d}:{est_seconds:02d}")

# Wait for all files to be saved
executor.shutdown(wait=True)

print(f"All chapters processed in {time_delta(initial_time)}")


## 5. Create Archive

After all chapters are processed, the folder is archived (ZIP) for easy download. Temporary files are deleted at the end to keep the workspace clean. This step produces a ready-to-use audiobook archive.

In [ ]:
import shutil
shutil.make_archive("Audiobook", "zip", "Audiobook")
print("Audiobook.zip has been created successfully.")

!rm -rf Audiobook
print("Audiobook folder and its content have been deleted.")